In [3]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from pathlib import Path

# =====================================
# 1. CSV 파일 불러오기
# =====================================

# 현재 작업 폴더 확인
current_dir = Path.cwd()
print("현재 작업 폴더:", current_dir)

# 현재 폴더 또는 data 폴더에서 CSV 찾기
if current_dir.name == "data":
    file_path = current_dir / "parking_2020.csv"
else:
    file_path = current_dir / "data" / "parking_2020.csv"

# 파일 존재 여부 확인
if not file_path.exists():
    raise FileNotFoundError(
        f"CSV 파일을 찾을 수 없습니다: {file_path}"
    )

df = pd.read_csv(file_path, encoding="utf-8-sig")

print("데이터 불러오기 성공!")
print(df.head())


# =====================================
# 2. 데이터 전처리
# =====================================

# 날짜 변환
df["기준일자"] = pd.to_datetime(
    df["기준일자"],
    errors="coerce"
)

# 위도와 경도 숫자로 변환
df["위도"] = pd.to_numeric(df["위도"], errors="coerce")
df["경도"] = pd.to_numeric(df["경도"], errors="coerce")

# 날짜와 좌표가 없는 행 제거
df = df.dropna(
    subset=["기준일자", "위도", "경도"]
)

# 요일 추출
df["요일"] = df["기준일자"].dt.dayofweek

# 요일 이름
weekdays = ["월", "화", "수", "목", "금", "토", "일"]


# =====================================
# 3. 지도 저장 폴더 생성
# =====================================

output_dir = current_dir / "요일별_지도"
output_dir.mkdir(exist_ok=True)


# =====================================
# 4. 요일별 지도 생성
# =====================================

for day in range(7):

    # 해당 요일 데이터만 추출
    day_df = df[df["요일"] == day].copy()

    print(f"{weekdays[day]}요일 데이터: {len(day_df)}건")

    # 대구 북구 중심 지도
    m = folium.Map(
        location=[35.89, 128.58],
        zoom_start=13
    )

    # 같은 장소와 좌표의 데이터 묶기
    grouped = (
        day_df.groupby(
            ["행정동", "위도", "경도"],
            dropna=False
        )
        .size()
        .reset_index(name="발생건수")
    )

    # 마커 군집 생성
    marker_cluster = MarkerCluster().add_to(m)

    # 장소별 마커 표시
    for _, row in grouped.iterrows():

        popup_text = f"""
        <b>요일:</b> {weekdays[day]}요일<br>
        <b>행정동:</b> {row['행정동']}<br>
        <b>발생 건수:</b> {row['발생건수']}건
        """

        folium.CircleMarker(
            location=[row["위도"], row["경도"]],
            radius=min(4 + row["발생건수"] ** 0.5, 15),
            popup=folium.Popup(popup_text, max_width=250),
            tooltip=f"{row['행정동']} ({row['발생건수']}건)",
            color="red",
            fill=True,
            fill_opacity=0.6
        ).add_to(marker_cluster)

    # HTML 파일 저장
    output_path = output_dir / f"{weekdays[day]}요일.html"
    m.save(str(output_path))

    print(f"{weekdays[day]}요일 지도 저장 완료!")

print("요일별 지도 7개 생성 완료!")

현재 작업 폴더: c:\Users\kdt66\Documents\18_kdt_01\18_kdt_01
데이터 불러오기 성공!
           경도         위도  행정동  위반상태        기준일자
0  128.605355  35.874821  칠성동  주간불법  2020-07-13
1  128.605307  35.874880  칠성동  주간불법  2020-07-13
2  128.605274  35.874910  칠성동  주간불법  2020-07-13
3  128.605156  35.874915  칠성동  주간불법  2020-07-13
4  128.605129  35.874911  칠성동  주간불법  2020-07-13
월요일 데이터: 9468건
월요일 지도 저장 완료!
화요일 데이터: 14763건
화요일 지도 저장 완료!
수요일 데이터: 14819건
수요일 지도 저장 완료!
목요일 데이터: 14538건
목요일 지도 저장 완료!
금요일 데이터: 13436건
금요일 지도 저장 완료!
토요일 데이터: 925건
토요일 지도 저장 완료!
일요일 데이터: 0건
일요일 지도 저장 완료!
요일별 지도 7개 생성 완료!
